# MedXIR: bge-reranker-v2-m3 over every chunk of the ensemble pool

Generated by `scripts/build_chunk_rerank_notebook.py`. Edit the generator, not this file.

RTX Pro 6000, Internet OFF. Writes `pool.parquet` (query_id, chunk_id, final_rank, final_score = bge) for
`fuse_scores.py --bge-pool`.

In [ ]:
import faulthandler
import json
import os
import resource
import time
import traceback
from contextlib import contextmanager
from pathlib import Path

OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)
T0 = time.perf_counter()
PROGRESS = OUT / 'progress.txt'
FAULT_FILE = open(OUT / 'faulthandler.txt', 'w')  # native crashes dump the Python stack here
faulthandler.enable(file=FAULT_FILE, all_threads=True)
FAILED = None


def note(message):
    rss_gib = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2 ** 20  # Linux reports KiB
    line = f'{time.perf_counter() - T0:8.1f}s  peak_rss={rss_gib:.1f}GiB  {message}'
    print(line, flush=True)
    with open(PROGRESS, 'a', encoding='utf-8') as file:
        file.write(line + '\n')


@contextmanager
def stage(name):
    # A Python exception is written to retrieval_error.txt and does not stop the kernel, so the
    # session ends COMPLETE and the files can be fetched; later steps are skipped.
    global FAILED
    note(f'start {name}')
    try:
        yield
        note(f'done {name}')
    except Exception:
        FAILED = name
        (OUT / 'retrieval_error.txt').write_text(f'stage: {name}\n\n{traceback.format_exc()}', encoding='utf-8')
        note(f'FAILED {name}; traceback in retrieval_error.txt')

## 1. Inputs, GPU check, chunk texts

In [ ]:
if FAILED is None:
    with stage('setup'):
        import glob
        import subprocess

        import numpy as np
        import pandas as pd
        import pyarrow as pa
        import pyarrow.compute as pc
        import pyarrow.parquet as pq
        import torch

        MAX_TOKENS = 512
        BATCH = 64


        def one_match(pattern, description):
            hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
            assert len(hits) == 1, f'{description}: expected one match, found {hits[:10]}'
            return Path(hits[0])


        POOL_PATH = one_match('pool.parquet', 'ensemble pool')
        QUERY_PATH = one_match('query.parquet', 'query.parquet')
        MODEL_PATH = one_match('bge-reranker-v2-m3/config.json', 'bge reranker').parent
        CHUNK_FILES = sorted(glob.glob('/kaggle/input/**/chunks/*/part-*.parquet', recursive=True))
        assert len(CHUNK_FILES) == 12, f'expected the 12 corpus v2 parts, found {len(CHUNK_FILES)}'
        print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'],
                             capture_output=True, text=True).stdout)
        gpu_name = torch.cuda.get_device_name(0)
        assert '6000' in gpu_name, f'Expected RTX Pro 6000, got {gpu_name!r}'
        pool = pd.read_parquet(POOL_PATH, columns=['query_id', 'chunk_id', 'final_rank'])
        query_text = {int(q): str(t) for q, t in pd.read_parquet(QUERY_PATH, columns=['id', 'query']).itertuples(index=False)}
        wanted = pa.array(sorted(set(pool['chunk_id'])))
        chunk_text = {}
        for path in CHUNK_FILES:
            table = pq.read_table(path, columns=['chunk_id', 'chunk_text'])
            table = table.filter(pc.is_in(table['chunk_id'], value_set=wanted))
            chunk_text.update(zip(table['chunk_id'].to_pylist(), table['chunk_text'].to_pylist()))
        missing = set(pool['chunk_id']) - set(chunk_text)
        assert not missing, f'{len(missing)} pool chunks not in the corpus, e.g. {sorted(missing)[:3]}'
        note(f'GPU {gpu_name}; {len(pool):,} pairs, {len(chunk_text):,} distinct chunks, {pool.query_id.nunique()} queries')

## 2. Score every (query, chunk) pair

In [ ]:
if FAILED is None:
    with stage('rerank'):
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        tok = AutoTokenizer.from_pretrained(str(MODEL_PATH))
        model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_PATH), torch_dtype=torch.bfloat16).cuda().eval()
        queries = [query_text[int(q)] for q in pool['query_id']]
        texts = [chunk_text[c] for c in pool['chunk_id']]
        order = np.argsort([len(t) for t in texts], kind='stable')   # similar lengths per batch: less padding
        scores = np.empty(len(texts), dtype=np.float32)
        t0, next_report = time.perf_counter(), 0.05
        with torch.inference_mode():
            for start in range(0, len(order), BATCH):
                index = order[start:start + BATCH]
                enc = tok([queries[i] for i in index], [texts[i] for i in index], padding=True, truncation=True,
                          max_length=MAX_TOKENS, return_tensors='pt').to('cuda')
                scores[index] = torch.sigmoid(model(**enc).logits.view(-1).float()).cpu().numpy()
                done = (start + len(index)) / len(order)
                if done >= next_report:
                    rate = (start + len(index)) / (time.perf_counter() - t0)
                    note(f'bge {done:.0%} | {rate:.0f} pairs/s | ETA {(len(order) - start) / rate / 60:.0f} min')
                    next_report += 0.05
        assert np.isfinite(scores).all()
        out = pool.assign(final_score=scores)
        out['final_rank'] = out.groupby('query_id')['final_score'].rank(ascending=False, method='first').astype(int) - 1
        out.sort_values(['query_id', 'final_rank']).to_parquet(OUT / 'pool.parquet', index=False, compression='zstd')
        note(f'pool.parquet written: {len(out):,} rows; bge median {float(np.median(scores)):.3f}')

## 3. Report

In [ ]:
note('finished; status ' + ('error' if FAILED else 'ok'))
if FAILED:
    (OUT / 'stage_status.json').write_text(json.dumps(
        dict(status='error', failed_stage=FAILED, error_file='retrieval_error.txt'), indent=2) + '\n')